# Two-state planar simulation from tabular data

This notebook demonstrates the complete data-to-simulation workflow:

1. Load a synthetic two-state cell dataset from CSV.
2. Hand-edit state-specific and global parameters in one cell.
3. Validate, run, and visualize a periodic planar simulation.

The CSV schema is deliberately small so it can later be replaced by experimental data. The planar engine uses state-dependent radii and proliferation rates rather than per-cell values, and the domain is periodic.

## 1. Imports and repository paths

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Work whether Jupyter starts in the repository root or notebooks/.
working_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (working_dir, *working_dir.parents) if (path / "cell_sphere_sim").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the cell-flow-sims repository root")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from cell_sphere_sim.planar import PlanarParams, PlanarSimulationEngine
from cell_sphere_sim.planar.neighbors import minimum_image_displacement
from cell_sphere_sim.state import StateTable

data_path = repo_root / "examples" / "data" / "synthetic_two_state_cells.csv"
print(f"Repository: {repo_root}")
print(f"Dataset:    {data_path}")

## 2. Load the example dataset

Required columns are `track_id`, `x`, `y`, `state_id`, `p_x`, and `p_y`. State IDs must be `0` or `1`. Polarity is normalized again before simulation, so small rounding differences in a CSV are harmless.

In [ ]:
cells = pd.read_csv(data_path)
required_columns = {"track_id", "x", "y", "state_id", "p_x", "p_y"}
missing = required_columns.difference(cells.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

print(f"Loaded {len(cells)} cells")
display(cells.head())
display(cells.groupby("state_id").size().rename("cell_count").to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for state, color, label in [(0, "tab:blue", "State 0"), (1, "tab:orange", "State 1")]:
    group = cells[cells["state_id"] == state]
    ax.scatter(group["x"], group["y"], s=55, color=color, label=label)
    ax.quiver(
        group["x"], group["y"], group["p_x"], group["p_y"],
        angles="xy", scale_units="xy", scale=1.8, width=0.004, color=color,
    )
ax.set(title="Loaded synthetic cells", xlabel="x", ylabel="y", aspect="equal")
ax.legend()
plt.show()

## 3. Hand-specify the parameters

Edit the values in the next cell, then rerun from that cell downward. Each row of `STATE_PARAMETERS` controls cells whose `state_id` matches the dictionary key.

- `R`: contact radius
- `Fm`: motility force
- `Dr`: rotational diffusion rate
- `fcil`: CIL relaxation rate
- `w`: adhesion parameter; pair adhesion is proportional to `w_i * w_j`
- `lambda_div`: unrestricted division rate (hazard) for that state
- `tau_div`: post-division motility pause for that state
- `gamma_s`: drag, so isolated speed is `Fm / gamma_s`
- `k_rep`: repulsive-force scale
- `alpha_dmin`: force-law regularization distance divided by `R_i + R_j`
- `division_separation_factor`: daughter separation divided by `R_a + R_b`; it is fixed here at `1.0`
- `division_projection_tolerance`: residual overlap allowed after instantaneous birth shoving

See `docs/PLANAR_2D.md` and `docs/PARAMETER_SCALING.md` before interpreting these as physical units. Daughter insertion uses an instantaneous, auditable geometric projection, but ordinary explicit-Euler mechanics are not protected by a global non-crossing constraint.

In [ ]:
# ---------------- EDIT PARAMETERS HERE ----------------
BOX_SIZE = (12.0, 9.0)  # (Lx, Ly), periodic

STATE_PARAMETERS = {
    0: {
        "name": "State 0",
        "R": 0.38,
        "Fm": 1.00,
        "Dr": 0.04,
        "fcil": 1.50,
        "w": 0.20,
        "lambda_div": 0.03,
        "tau_div": 0.50,
    },
    1: {
        "name": "State 1",
        "R": 0.45,
        "Fm": 0.70,
        "Dr": 0.08,
        "fcil": 2.50,
        "w": 0.30,
        "lambda_div": 0.06,
        "tau_div": 0.80,
    },
}

GLOBAL_PARAMETERS = {
    "gamma_s": 1.0,
    "k_rep": 2.0,
    "alpha_dmin": 0.2,
    "eps": 1e-8,
    "dt": 0.01,
    "neighbor_radius_buffer": 0.1,
    "division_separation_factor": 1.0,
    "division_projection_enabled": True,
    "division_projection_tolerance": 1e-8,
    "division_projection_max_iterations": 500,
}

RUN_PARAMETERS = {
    "seed": 2026,
    "n_steps": 500,
    "record_every": 5,
}
# ------------------------------------------------------

state_parameter_table = pd.DataFrame.from_dict(STATE_PARAMETERS, orient="index")
state_parameter_table.index.name = "state_id"
display(state_parameter_table)
display(pd.Series(GLOBAL_PARAMETERS, name="value").to_frame())
display(pd.Series(RUN_PARAMETERS, name="value").to_frame())

## 4. Validate the data and build the two-state engine

In [ ]:
numeric_columns = ["x", "y", "state_id", "p_x", "p_y"]
if not np.all(np.isfinite(cells[numeric_columns].to_numpy(dtype=float))):
    raise ValueError("Input data contains non-finite coordinates, states, or polarities")
if cells["track_id"].duplicated().any():
    raise ValueError("track_id values must be unique at initialization")
if not set(cells["state_id"].astype(int)).issubset(STATE_PARAMETERS):
    raise ValueError("Every state_id must have a row in STATE_PARAMETERS")

x0 = cells[["x", "y"]].to_numpy(dtype=float)
p0 = cells[["p_x", "p_y"]].to_numpy(dtype=float)
state_id = cells["state_id"].to_numpy(dtype=np.int32)
track_id = cells["track_id"].to_numpy(dtype=np.int64)

box = np.asarray(BOX_SIZE, dtype=float)
if np.any(x0 < 0.0) or np.any(x0 >= box):
    raise ValueError("All input positions must lie inside [0,Lx) x [0,Ly)")
p_norm = np.linalg.norm(p0, axis=1)
if np.any(p_norm <= GLOBAL_PARAMETERS["eps"]):
    raise ValueError("Every input polarity must have non-zero length")
p0 = p0 / p_norm[:, None]

state_keys = sorted(STATE_PARAMETERS)
if state_keys != [0, 1]:
    raise ValueError("This notebook expects exactly two contiguous states: 0 and 1")

state_table = StateTable(
    R=np.array([STATE_PARAMETERS[key]["R"] for key in state_keys], dtype=float),
    Fm=np.array([STATE_PARAMETERS[key]["Fm"] for key in state_keys], dtype=float),
    Dr=np.array([STATE_PARAMETERS[key]["Dr"] for key in state_keys], dtype=float),
    fcil=np.array([STATE_PARAMETERS[key]["fcil"] for key in state_keys], dtype=float),
    w=np.array([STATE_PARAMETERS[key]["w"] for key in state_keys], dtype=float),
    lambda_div=np.array([STATE_PARAMETERS[key]["lambda_div"] for key in state_keys], dtype=float),
    tau_div=np.array([STATE_PARAMETERS[key]["tau_div"] for key in state_keys], dtype=float),
)
params = PlanarParams(
    box_size=BOX_SIZE,
    gamma_s=GLOBAL_PARAMETERS["gamma_s"],
    k_rep=GLOBAL_PARAMETERS["k_rep"],
    alpha_dmin=GLOBAL_PARAMETERS["alpha_dmin"],
    eps=GLOBAL_PARAMETERS["eps"],
    dt=GLOBAL_PARAMETERS["dt"],
    neighbor_radius_buffer=GLOBAL_PARAMETERS["neighbor_radius_buffer"],
    division_enabled=True,
    division_separation_factor=GLOBAL_PARAMETERS["division_separation_factor"],
    division_projection_enabled=GLOBAL_PARAMETERS["division_projection_enabled"],
    division_projection_tolerance=GLOBAL_PARAMETERS["division_projection_tolerance"],
    division_projection_max_iterations=GLOBAL_PARAMETERS["division_projection_max_iterations"],
)

# Report the closest periodic pair before running.
closest = np.inf
closest_pair = None
for i in range(len(x0)):
    displacement = minimum_image_displacement(x0[i], x0[i + 1 :], box)
    distances = np.linalg.norm(displacement, axis=1)
    if distances.size and distances.min() < closest:
        offset = int(np.argmin(distances))
        closest = float(distances[offset])
        closest_pair = (i, i + 1 + offset)
print(f"Closest initial periodic pair: {closest_pair}, distance={closest:.3f}")

rng = np.random.default_rng(RUN_PARAMETERS["seed"])
engine = PlanarSimulationEngine(
    x=x0,
    p=p0,
    state_id=state_id,
    state_vars=np.zeros((len(cells), 0), dtype=float),
    state_table=state_table,
    params=params,
    rng=rng,
    track_id=track_id,
)
print(f"Engine ready with {len(engine.x)} cells and initial track IDs {track_id.min()}–{track_id.max()}")

## 5. Run the simulation

The cell below records variable-length positions, states, and lineage IDs every `record_every` steps. Division is unrestricted: it is not suppressed by crowding or a carrying capacity. Rerunning the build cell first resets positions and the random generator, making the run reproducible.

In [ ]:
n_steps = int(RUN_PARAMETERS["n_steps"])
record_every = int(RUN_PARAMETERS["record_every"])
if n_steps < 1 or record_every < 1:
    raise ValueError("n_steps and record_every must be positive")

records = [{
    "step": 0, "t": 0.0, "x": engine.x.copy(), "p": engine.p.copy(),
    "state_id": engine.state_id.copy(), "track_id": engine.track_id.copy(),
    "parent_id": engine.parent_id.copy(),
}]
diagnostic_rows = []

for step in range(n_steps):
    t = step * float(engine.params.dt)
    diagnostics = engine.step(t)
    state_counts = np.bincount(engine.state_id, minlength=2)
    diagnostic_rows.append({
        "step": step + 1, "t": t + float(engine.params.dt), **diagnostics,
        "state_0_cells": int(state_counts[0]), "state_1_cells": int(state_counts[1]),
    })
    if (step + 1) % record_every == 0 or step + 1 == n_steps:
        records.append(
            {
                "step": step + 1, "t": t + float(engine.params.dt),
                "x": engine.x.copy(), "p": engine.p.copy(),
                "state_id": engine.state_id.copy(), "track_id": engine.track_id.copy(),
                "parent_id": engine.parent_id.copy(),
            }
        )
    if not all(np.all(np.isfinite(array)) for array in (engine.x, engine.p, engine.v)):
        raise FloatingPointError(f"Non-finite simulation state at step {step + 1}")

diagnostics_df = pd.DataFrame(diagnostic_rows)
print(f"Completed {n_steps} steps; recorded {len(records)} snapshots")
projection_steps = diagnostics_df[diagnostics_df["n_divisions"] > 0]
if not projection_steps.empty:
    print(
        "Largest division shove: "
        f"{int(projection_steps['division_projection_cells_moved'].max())} cells, "
        f"{projection_steps['division_projection_max_displacement'].max():.4g} distance"
    )
display(diagnostics_df.tail(1).T.rename(columns={diagnostics_df.index[-1]: "final"}))

## 6. Visualize the initial and final states

In [ ]:
state_colors = np.array(["tab:blue", "tab:orange"])
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for ax, snapshot, title in [
    (axes[0], records[0], "Initial data"),
    (axes[1], records[-1], f"After {n_steps} steps"),
]:
    snapshot_sizes = 900.0 * state_table.R[snapshot["state_id"]] ** 2
    for state in (0, 1):
        mask = snapshot["state_id"] == state
        ax.scatter(
            snapshot["x"][mask, 0], snapshot["x"][mask, 1],
            s=snapshot_sizes[mask], color=state_colors[state], alpha=0.75,
            edgecolor="black", linewidth=0.4, label=STATE_PARAMETERS[state]["name"],
        )
        ax.quiver(
            snapshot["x"][mask, 0], snapshot["x"][mask, 1],
            snapshot["p"][mask, 0], snapshot["p"][mask, 1],
            angles="xy", scale_units="xy", scale=2.0,
            width=0.004, color=state_colors[state],
        )
    ax.set(xlim=(0, box[0]), ylim=(0, box[1]), xlabel="x", ylabel="y", title=title)
    ax.set_aspect("equal", adjustable="box")
    ax.legend(loc="upper right")
plt.show()

## 7. Inspect collective diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
metric_panels = [
    ("mean_speed", "Mean speed"),
    ("mean_contacts", "Mean contacts"),
    ("polarization", "Polarization"),
]
for ax, (column, title) in zip(axes.flat[:3], metric_panels):
    ax.plot(diagnostics_df["t"], diagnostics_df[column], lw=1.5)
    ax.set(title=title, xlabel="time", ylabel=column)
    ax.grid(alpha=0.25)
population_ax = axes.flat[3]
population_ax.plot(diagnostics_df["t"], diagnostics_df["n_cells"], label="Total")
population_ax.plot(diagnostics_df["t"], diagnostics_df["state_0_cells"], label="State 0")
population_ax.plot(diagnostics_df["t"], diagnostics_df["state_1_cells"], label="State 1")
population_ax.set(title="Population", xlabel="time", ylabel="cells")
population_ax.grid(alpha=0.25)
population_ax.legend()
plt.show()

## 8. Optional: prepare final output for export

This creates a table with final daughter track IDs, parent IDs, and the final planar state. Uncomment the final line to write it to `notebooks/outputs/`.

In [ ]:
final_cells = pd.DataFrame(
    {
        "track_id": engine.track_id,
        "parent_id": engine.parent_id,
        "x": engine.x[:, 0],
        "y": engine.x[:, 1],
        "state_id": engine.state_id,
        "p_x": engine.p[:, 0],
        "p_y": engine.p[:, 1],
        "v_x": engine.v[:, 0],
        "v_y": engine.v[:, 1],
    }
)
display(final_cells.head())

output_path = repo_root / "notebooks" / "outputs" / "two_state_final.csv"
# final_cells.to_csv(output_path, index=False)
# print(f"Saved {output_path}")